In [0]:
%run ./_utils

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

#### Create `openalex.sources.openalex_sources_snapshot` in same format as API

In [ ]:
df_transformed = (
    spark.read.table("openalex.sources.sources_api")
    # Prefix numeric ID with full URL
    .withColumn("id", F.concat(F.lit("https://openalex.org/S"), F.col("id")))
    # Coalesce null arrays to empty arrays
    .withColumn("issn", F.coalesce(F.col("issn"), F.array()))
    .withColumn("host_organization_lineage", F.coalesce(F.col("host_organization_lineage"), F.array()))
    .withColumn("apc_prices", F.coalesce(F.col("apc_prices"), F.array()))
    .withColumn("societies", F.coalesce(F.col("societies"), F.array()))
    .withColumn("alternate_titles", F.coalesce(F.col("alternate_titles"), F.array()))
    .withColumn("topics", F.coalesce(F.col("topics"), F.array()))
    .withColumn("topic_share", F.coalesce(F.col("topic_share"), F.array()))
    .withColumn("counts_by_year", F.coalesce(F.col("counts_by_year"), F.array()))
)

df_transformed.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("openalex.sources.openalex_sources_snapshot")

#### Export to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/sources/`.

In [0]:
date_str = get_snapshot_date()
df = spark.read.table("openalex.sources.openalex_sources_snapshot")
export_partitioned_all_formats(spark, dbutils, df, date_str, "sources", salt=False)